In [7]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict
from google import genai
import os
from dotenv import load_dotenv

In [8]:
load_dotenv()
api_key = os.getenv("GOOGLE_GEMINI_API")

In [9]:
client = genai.Client(api_key=api_key)

In [10]:
class LLMState(TypedDict):
    title : str
    outline : str
    content : str

In [11]:
def create_outline(State : LLMState)-> LLMState:

    title = State['title']
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents= prompt
        )
    State['outline'] = outline

    return State

In [12]:
def create_blog(State : LLMState) -> LLMState:
    title = State['title']
    outline = State['outline']

    prompt = f"Write a detailed blog on the title - {title} using the following outline - {outline}"
    content = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = prompt
    )
    State['content'] = content

    return State

In [13]:
graph = StateGraph(LLMState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog",create_blog)

graph.add_edge(START,"create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()

In [15]:
intial_state = {'title' : 'Rise of AI in INDIA'}
final_state = workflow.invoke(intial_state)
print(final_state)

{'title': 'Rise of AI in INDIA', 'outline': GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in India."** 

---

# Blog Title Options:
*   *The AI Takeover in India: How Artificial Intelligence is Shaping the Nation’s Future*
*   *From Jugaad to GenAI: The Meteoric Rise of Artificial Intelligence in India*
*   *Bharat 2.0: How India is Becoming a Global AI Powerhouse*

---

## I. Introduction
*   **The Hook:** Briefly paint a picture of India’s digital transformation (UPI, massive smartphone penetration, cheap data). 
*   **The Shift:** Transition from India as an IT back-office to India as an AI innovator.
*   **Thesis Statement:** Powered by a massive talent pool, government initiatives, and unique societal challenges, India is rapidly emerging as a global leader in the Art